In [ ]:
import sys
sys.path.insert(0, '../')

print(sys.path)

In [ ]:
import obsidian
import pandas as pd
import numpy as np
print(f'obsidian version: ' + obsidian.__version__)

from obsidian.experiment import AdvExpDesigner
from obsidian.experiment.sampling import sample_with_bias, best_sample

In [ ]:
#generate random data for this demo
np.random.seed(42)

n = 1000
demo_data = pd.DataFrame({
    'reagent_conc': np.round(np.random.uniform(0.1, 1.0, n), 2),
    'ionic_strength': np.round(np.random.uniform(10, 100, n), 2),
    'surfactant_conc': np.round(np.random.uniform(0.01, 0.2, n), 3),
    'compound_A': np.round(np.random.uniform(0, 50, n), 2),
    'compound_B': np.round(np.random.uniform(0, 50, n), 2),
    'sugar': np.random.choice(['glucose', 'fructose', 'sucrose'], n),
    'surfactant': np.random.choice(['SDS', 'Tween20', 'TritonX'], n),
    'buffer': np.random.choice(['PBS', 'Tris', 'HEPES'], n),
    'pH': np.round(np.random.uniform(5.5, 8.5, n), 2)
})

demo_data.index.name = 'FormulationID'
demo_data

In [ ]:
#Initialize existing experimental data as an AdvExpDesigner object
designer = AdvExpDesigner(design_df=demo_data)

In [ ]:
"""
You can sample an existing dataset with or without bias: 
Bias dictionary format : {"column": [lower_bound, upper_bound, relative_weight]}

- Weight >1 increases sampling probability for in-range rows.
- Weight <1 decreases it.
- Weight = 0 excludes those rows entirely.
"""

bias = {
    "ionic_strength": [50, 60, 3.0], 
}

seed = np.random.randint(0,1000)
print(f"Random seed for reproducibility: {seed}")

#We can easily create a random sample of n samples with weights using built in Pandas functions
#enforce = True allows you to force the boundary to be true ; resultant sample may not be space-filling.
sample = sample_with_bias(designer.design, n=1000, replace=False, seed=seed, bias=bias, plot_weights=True, enforce=False)

sample

In [ ]:
#One-hot encode your categorical columns for easy handling in determining Euclidean distance
df_encoded = pd.get_dummies(designer.design, columns=["sugar", "surfactant", "buffer"], dtype=int) 

In [ ]:
"""
perform random sampling n_trial times, select the best one via criteria metric:
metric:
    - "maximin":   maximize the minimum pairwise Euclidean distance
    - "mean_nn":   maximize the mean nearest-neighbor Euclidean distance
    - "hybrid":    0.6*maximin + 0.4*mean_nn 
"""
seed = np.random.randint(0,1000)
print(f"Random seed for reproducibility: {seed}")

optimal_sample, info = best_sample(
    df_encoded, 10, feature_cols=df_encoded.columns, n_trials=1000,
    bias=bias, plot_weights=True, enforce=False, random_state=seed, metric="hybrid"
)

print(info)
optimal_sample


In [ ]:
#decode from one-hot encoding
normal_cols = list(optimal_sample.columns)[0:6]
encoded_cols = list(optimal_sample.columns)[6:]
decoded = pd.from_dummies(optimal_sample[encoded_cols],sep="_")
optimal_design_decoded = pd.concat([optimal_sample[normal_cols], decoded], axis=1)
optimal_design_decoded

In [ ]:
print(designer.plot_histograms(optimal_design_decoded))